# Quickstart

Runs anywhere with no data download: a seeded synthetic market is built in. Install with `pip install -e '.[notebooks,app]'`.

In [ ]:
import stratbt.metrics as metrics
from stratbt import run_backtest, SyntheticProvider, ExecutionModel, RiskLimits
from stratbt.strategy import registry

for key, spec in registry.available().items():
    print(f'{key:16s} {spec.label}')

## Run a strategy

`ExecutionModel.realistic()` = next-second fills, a bid/ask spread and per-contract commission.

In [ ]:
market = SyntheticProvider(n_days=21, seed=7, iv_premium=0.10)   # options 10% richer than realised vol
result = run_backtest('short_strangle', market, params={'width': 3},
                      execution=ExecutionModel.realistic(),
                      risk=RiskLimits(max_daily_loss=2_000))
metrics.summary_table(result).T

In [ ]:
metrics.plot_cumulative_pnl(result);

## Write your own

Subclass `BaseStrategy`, then register it so the CLI and dashboard pick it up.

In [ ]:
import numpy as np
from stratbt import BaseStrategy, Order, StrategySpec, ParamSpec, register
from stratbt.instruments import Instrument
from stratbt.strategy.base import nearest_strike

class OpenAndHold(BaseStrategy):
    """Buy one ATM call at `entry_delay_s`, hold to the close (the engine flattens)."""
    def __init__(self, underlier, expiry, strikes, entry_delay_s=600):
        self.u, self.e, self.k, self.delay, self.done = underlier, expiry, strikes, entry_delay_s, False

    def on_second(self, ts, futures_price, option_price, holdings):
        if self.done or (ts - ts.normalize()).total_seconds() < 9.5 * 3600 + self.delay:
            return []
        call = Instrument(self.u, self.e, nearest_strike(self.k, futures_price), 'CE')
        if option_price(call) is None:
            return []
        self.done = True
        return [Order(call, 'BUY')]

register(StrategySpec('open_and_hold', 'Open and Hold', OpenAndHold, 'Buy an ATM call and hold.',
                      [ParamSpec('entry_delay_s', 600, 0, 3600, 60, 'Entry delay (s)')]))
run_backtest('open_and_hold', market, underliers=['SPY']).daily_pnl.head()